# Part 1 — What actually makes an agent "deep"
### Competitive Market Intelligence Analyst

A normal agent does **one task in one shot**. Give it a document, it calls a few tools, it answers.
Ask it to research six competitors and write a 20-page report and it falls apart — it runs out of
context, forgets what it already did, and has nowhere to put intermediate work.

A **deep agent** fixes exactly those three problems. It is a normal agent plus four things:

| Capability | Built-in tool | Problem it solves |
|---|---|---|
| **Planning** | `write_todos` | Keeps a visible plan so it doesn't lose the thread on a long task |
| **Filesystem** | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` | Offloads work to disk instead of holding everything in context |
| **Shell** | `execute` | Runs commands inside the sandbox |
| **Delegation** | `task` | Spawns subagents with their own clean context (Part 2) |
| **Skills** | folders with a `SKILL.md` | Loads detailed instructions only when relevant (Part 3) |

You write **none** of those. `create_deep_agent` supplies the harness; you supply domain tools.

---

### 📌 Why `deepagents` is pinned to `0.4.11`

This library moves fast and its defaults change between minor versions. Two that bite:

**1. `write_todos` disappears on 0.7.x.** On 0.4.11 it is included automatically. On 0.7.x it is
only auto-added for the OpenAI Codex model profile — with an Anthropic model your agent silently
has no planning tool and *nothing warns you*. If you ever upgrade:

```python
from langchain.agents.middleware import TodoListMiddleware
create_deep_agent(..., middleware=[TodoListMiddleware()])
```

**2. `FilesystemBackend(virtual_mode=...)` changed default in 0.5.0.** See the warning in Part 1.

Pinning an exact version (`deepagents==0.4.11`, not `>=`) is what makes this reproducible. A `>=`
constraint silently gives a future reader a different agent than the one you tested.

The habit that catches this whole class of bug: **print the tool list of every agent you build.**
We define `show_tools()` below and use it every time.

> **This notebook covers Parts 1–2.** Part 1 = the basic deep agent, Part 2 = subagents.
> Then `3_skills_and_critic.ipynb`, then the Streamlit app.

---
## Setup

Make sure the kernel in the top-right says **`Python (agents-projects)`**.

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

from research_tools import ensure_sandbox, internet_search, show_tree

SANDBOX = ensure_sandbox()
print(f"Agent sandbox: {SANDBOX}")

# Deep agents run MANY model calls in a loop, so model choice matters for both
# quality and cost.
MODEL = "anthropic:claude-sonnet-5"    # good balance -- recommended default
# MODEL = "anthropic:claude-haiku-4-5" # cheapest, fine for testing the plumbing
# MODEL = "anthropic:claude-opus-5"    # best reasoning, most expensive
print(f"Model: {MODEL}")

In [ ]:
# Sanity-check the one custom tool we give the agent.
results = internet_search.invoke({"query": "Zomato market share India food delivery 2026", "max_results": 2})

for r in results["results"]:
    print(f"- {r['title']}\n  {r['url']}\n  {r['content'][:160]}...\n")

---
## A helper to watch the agent think

Deep agents run for minutes and make dozens of calls. `.invoke()` gives you a black box; `.stream()`
lets you watch. **Build this habit** — it is how you debug agents.

Look for `write_todos` firing early. That's the agent planning before it acts.

In [ ]:
def run_agent(agent, prompt: str, recursion_limit: int = 120):
    """Stream an agent run, printing every tool call as it happens."""
    final = None
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": recursion_limit},
        stream_mode="values",
    ):
        final = chunk
        msg = chunk["messages"][-1]

        for call in getattr(msg, "tool_calls", None) or []:
            name, args = call["name"], call["args"]
            if name == "write_todos":
                print("\n  PLAN:")
                for todo in args.get("todos", []):
                    mark = {"completed": "x", "in_progress": ">"}.get(todo.get("status"), " ")
                    print(f"    [{mark}] {todo.get('content')}")
            elif name == "task":
                print(f"  DELEGATE -> {args.get('subagent_type')}: {str(args.get('description'))[:70]}")
            elif name == "internet_search":
                print(f"  SEARCH   -> {args.get('query')}")
            elif name in ("write_file", "edit_file"):
                print(f"  {name.upper():8} -> {args.get('file_path')}")
            else:
                print(f"  {name:8} -> {str(args)[:70]}")
    return final

---
## Part 1 — Your first deep agent

Three arguments do all the work:

- **`tools=[internet_search]`** — the only tool *we* wrote.
- **`backend=FilesystemBackend(root_dir=SANDBOX, virtual_mode=True)`** — where its files really live.
- **`system_prompt=...`** — crucially, this must **tell it to use the filesystem**. A deep agent
  that isn't told to write files will just answer in chat and you gain nothing.

### ⚠️ Always pass `virtual_mode=True` explicitly on 0.4.11

On this version `virtual_mode` defaults to `None`, which behaves as **`False`** and emits a
`DeprecationWarning`. That matters for safety:

| | `virtual_mode=False` (the 0.4.11 default) | `virtual_mode=True` (what we use) |
|---|---|---|
| `/output/x.md` | writes to your **real filesystem root** | writes to `sandbox/output/x.md` |
| `../../secrets` | **escapes the sandbox** | blocked |

So the agent thinks it's writing to `/output/report.md`; it actually lands in
`sandbox/output/report.md`, and it cannot wander into the rest of your machine. Tutorials that omit
this argument are relying on a default that changed in 0.5.0 — always be explicit.

*(This is guardrail, not true isolation — it's not a container. Don't hand an agent tools you'd be
unhappy to see misused.)*

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

RESEARCHER_PROMPT = """You are a competitive market intelligence analyst.

## How you work
1. FIRST call write_todos to lay out your plan. Update it as you go.
2. Research with internet_search. Use SPECIFIC queries -- "Swiggy Q3 FY2026 revenue"
   beats "Swiggy financials". Run several searches from different angles.
3. Save raw findings to /research/<topic>.md as you gather them. Do NOT try to hold
   everything in your head -- that is what the filesystem is for.
4. Write your final deliverable to /output/.

## Rules
- Every factual claim needs a source URL. No URL, no claim.
- If you cannot verify something, write "unverified" rather than guessing.
- Distinguish clearly between reported facts and your own analysis.
"""

basic_agent = create_deep_agent(
    model=MODEL,
    tools=[internet_search],
    system_prompt=RESEARCHER_PROMPT,
    backend=FilesystemBackend(root_dir=SANDBOX, virtual_mode=True),
)


def show_tools(agent, label):
    """ALWAYS do this after building an agent. Never assume a tool is present."""
    names = sorted({t.name for t in agent.nodes["tools"].bound._tools_by_name.values()})
    print(f"{label} exposes {len(names)} tools:")
    print("   " + ", ".join(names))
    return names


names = show_tools(basic_agent, "basic_agent")

print("\n  we wrote:  internet_search")
print("  free:      everything else")
print(f"\n  write_todos present? {'write_todos' in names}   <- automatic on 0.4.11")

In [ ]:
# This takes a couple of minutes. Watch the PLAN appear first, then the searches.
result = run_agent(
    basic_agent,
    "Research Zepto, the Indian quick-commerce company. Cover: business model, "
    "funding and valuation, main competitors, and their biggest strategic risk. "
    "Save your findings to /research/zepto.md and a 1-page brief to /output/zepto_brief.md.",
)

print("\n" + "=" * 70)
print(result["messages"][-1].content[:1500])

### The payoff — it produced real files

This is the difference from Project 1. The agent didn't just talk; it left artefacts on disk.

In [ ]:
print("sandbox/")
show_tree()

In [ ]:
from IPython.display import Markdown, display

brief = SANDBOX / "output" / "zepto_brief.md"
display(Markdown(brief.read_text(encoding="utf-8"))) if brief.exists() else print("not found -- check the tree above for the actual filename")

---
## The context problem — why Part 2 exists

Now try the *real* task: not one company, but a whole competitive landscape.

Run the cell below and watch what happens. **It will work, but badly.** Count the searches — every
single result stays in the agent's context window. By competitor four it is carrying thousands of
tokens of Blinkit research while trying to think about Swiggy Instamart. Quality degrades, cost
climbs, and it starts forgetting its own plan.

> Feel free to skip running this one — it costs real money and the point is the diagnosis, not the
> output. Read on if you'd rather not spend it.

In [ ]:
result = run_agent(
    basic_agent,
    "Now compare Zepto against Blinkit and Swiggy Instamart. For EACH company research "
    "business model, funding, market share, and strategic risks. Then write a comparison "
    "to /output/quick_commerce_comparison.md.",
)

n_msgs = len(result["messages"])
print(f"\n{n_msgs} messages accumulated in ONE context window.")
print("Every search result above is still sitting in it.")

---
## Part 2 — Subagents fix this

The fix is **context quarantine**. Instead of one agent researching all three companies, a *lead*
agent delegates one company to each subagent via the built-in `task` tool.

```
                    lead agent  (holds only the plan + 3 short summaries)
                         |
        +----------------+----------------+
        |                |                |
  competitor-        competitor-     competitor-
  researcher         researcher      researcher
   (Zepto)            (Blinkit)      (Instamart)
   own context        own context     own context
```

Each subagent burns its own context on searches, writes its findings to a file, and returns a short
summary. **The lead never sees the raw search results at all** — it sees three paragraphs and three
filenames.

A subagent is just a dict. These are the fields (from the installed `SubAgent` type):

| Field | Required | Purpose |
|---|---|---|
| `name` | yes | How the lead refers to it |
| `description` | yes | **The lead reads this to decide when to delegate.** Write it carefully. |
| `system_prompt` | yes | The subagent's own instructions |
| `tools` | no | Defaults to the lead's tools |
| `model` | no | Can differ! Cheap model for grunt work, strong model for analysis |

In [ ]:
competitor_researcher = {
    "name": "competitor-researcher",
    "description": (
        "Researches ONE company in depth and writes a findings file. "
        "Delegate one call per company -- never ask it about two companies at once. "
        "Returns a short summary plus the path to the file it wrote."
    ),
    "system_prompt": """You research exactly ONE company, thoroughly.

Run at least 4 internet_search calls from different angles:
  - business model and revenue streams
  - funding, valuation, financial performance
  - market position and share
  - risks, controversies, competitive threats

Write everything to /research/<company_lowercase>.md with this structure:
  # <Company>
  ## Business model
  ## Financials & funding
  ## Market position
  ## Risks
  ## Sources        <- every URL you used

Every claim needs a source URL next to it. Write "unverified" if you could not confirm it.

Your FINAL REPLY must be short: 5-8 bullet points of the key findings, then the file path.
Do not paste the full research into your reply -- it is in the file.""",
    "model": MODEL,
}

print(competitor_researcher["name"], "defined")

In [ ]:
LEAD_PROMPT = """You are the lead analyst on a competitive intelligence team.

## Your job is to COORDINATE, not to research
You have a `competitor-researcher` subagent. Use it.

1. Call write_todos first with one item per company, plus a final synthesis item.
2. Delegate ONE company per `task` call to competitor-researcher.
   Send them all in the same turn so they run concurrently.
3. Do NOT call internet_search yourself for company facts -- that is the subagents' job.
   Delegating keeps your own context clean, which is the entire point.
4. When they report back, read their files with read_file, then write the comparison
   to /output/.

## The comparison must contain
- A side-by-side table (business model, funding, market position, key risk)
- 3-5 findings that only become visible when you compare them
- A "who is best positioned and why" call, with your reasoning
- Every source URL
"""

lead_agent = create_deep_agent(
    model=MODEL,
    tools=[internet_search],
    system_prompt=LEAD_PROMPT,
    subagents=[competitor_researcher],
    backend=FilesystemBackend(root_dir=SANDBOX, virtual_mode=True),
)

show_tools(lead_agent, "lead_agent")
print("\n'task' is how it delegates to competitor-researcher.")

In [ ]:
# Watch for DELEGATE lines. That is the lead handing work off.
result = run_agent(
    lead_agent,
    "Analyse the Indian quick-commerce market. Compare Zepto, Blinkit, and Swiggy Instamart. "
    "Write the comparison to /output/quick_commerce_analysis.md.",
    recursion_limit=200,
)

print(f"\n{len(result['messages'])} messages in the LEAD's context.")
print("Compare that to the single-agent run above -- the searches happened inside")
print("the subagents, so they never touched this context window.")

In [ ]:
print("sandbox/")
show_tree()

In [ ]:
report = SANDBOX / "output" / "quick_commerce_analysis.md"
display(Markdown(report.read_text(encoding="utf-8"))) if report.exists() else show_tree()

---
## What you just learned

1. **`create_deep_agent` = `create_agent` + planning + filesystem + delegation.** You wrote one tool;
   the harness gave you eight more.
2. **The filesystem is a context-management strategy, not a feature.** Writing findings to disk is how
   the agent stays coherent across a long task.
3. **Subagents quarantine context.** The lead coordinates on a clean context while subagents do the
   token-heavy work. This is the single most important idea in the whole library.
4. **A subagent's `description` is a prompt.** The lead reads it to decide when to delegate — a vague
   description means the lead does the work itself and you lose the benefit.

### Try it yourself
- Set the subagent's `"model"` to `"anthropic:claude-haiku-4-5"` and keep the lead on Sonnet. Mixed-model
  teams are a real cost lever — cheap workers, smart coordinator.
- Change `description` to just `"researches things"` and re-run. Watch the lead stop delegating.
- Point it at a market you actually know and judge the output quality yourself.

### Next
`3_skills_and_critic.ipynb` — Agent Skills (progressive disclosure) and a fact-checking critic loop
that sends the report back for revision when claims aren't sourced.